In [1]:
library(limma)
data_dir <- '/home/ethan-xiao/food-allergy-biomarkers/data'

#Reloading necessary data
mVals_189148_shared <- readRDS(file.path(data_dir, 'mVals_189148_shared.rds'))
allergy_status <- readRDS(file.path(data_dir, 'allergy_status.rds'))
batch <- readRDS(file.path(data_dir, 'batch.rds'))
scan_dates <- readRDS(file.path(data_dir, 'scan_dates_189148.rds'))

status_189148 <- allergy_status[batch == 'GSE189148']
design_189148 <- model.matrix(~ status_189148 + factor(scan_dates))
fit_189148 <- eBayes(lmFit(mVals_189148_shared, design_189148))
tt_189148 <- topTable(fit_189148, coef = 2, number = Inf, sort.by = 'none')

tt_189148[c('cg11090352','cg15076659','cg08469540'),
          c('logFC','P.Value','adj.P.Val')]

,logFC,P.Value,adj.P.Val
,<dbl>,<dbl>,<dbl>
cg11090352,0.08242869,0.07682545,0.9999969
cg15076659,0.04219258,0.66980577,0.9999969
cg08469540,-0.37996349,0.05487312,0.9999969


Check ISG15 nearby probes

In [2]:
BiocManager::install("IlluminaHumanMethylationEPICanno.ilm10b4.hg19")
library(minfi)
library(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)
ann <- getAnnotation(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03-11)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'IlluminaHumanMethylationEPICanno.ilm10b4.hg19'”
Old packages: 'bit64', 'bitops', 'curl', 'data.table', 'DelayedArray', 'httr',
  'lattice', 'Matrix', 'nlme', 'RCurl', 'S4Vectors', 'SparseArray', 'stringi',
  'survival', 'XML'

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:limma’:

    plotMA


The following ob

In [3]:
mVals_114134_shared <- readRDS(file.path(data_dir, 'mVals_114134_shared.rds'))
status_114134 <- allergy_status[batch == 'GSE114134']
fit_114134 <- eBayes(lmFit(mVals_114134_shared, model.matrix(~ status_114134)))
tt_114134 <- topTable(fit_114134, coef = 2, number = Inf, sort.by = 'none')

dim(tt_114134)

[1] 751107      6

In [4]:
saveRDS(tt_114134, file.path(data_dir, 'tt_114134.rds'))
saveRDS(tt_189148, file.path(data_dir, 'tt_189148.rds'))

In [5]:
ann <- getAnnotation(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)
isg15 <- rownames(ann)[grepl('ISG15', ann$UCSC_RefGene_Name)]
isg15 <- intersect(isg15, rownames(tt_189148))

data.frame(
  probe = isg15,
  pos   = ann[isg15, 'pos'],
  island = ann[isg15, 'Relation_to_Island'],
  lfc_infant = tt_114134[isg15, 'logFC'],
  p_infant   = tt_114134[isg15, 'P.Value'],
  lfc_adol   = tt_189148[isg15, 'logFC'],
  p_adol     = tt_189148[isg15, 'P.Value']
)

probe,pos,island,lfc_infant,p_infant,lfc_adol,p_adol
<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
cg21503301,948814,Island,-0.21333937,8.724205e-02,0.173497639,0.00266151
cg20022511,948740,Island,-0.05222767,3.505424e-01,0.075081781,0.28185493
cg22115423,948819,Island,-0.08331560,1.216829e-01,0.113825462,0.20255527
cg08067365,948893,Island,-0.08882691,1.204951e-01,-0.116914446,0.07678694
cg04788999,949850,Island,-0.05365011,3.885802e-01,0.109688496,0.32035220
cg20062691,949392,Island,-0.08367150,2.488450e-01,0.092598988,0.36894681
cg23684711,948879,Island,-0.01089672,7.915002e-01,-0.028815062,0.69245430
cg11211792,949634,Island,-0.06119426,4.036039e-01,0.044092841,0.68305080
cg22182593,947864,N_Shore,0.03273700,5.918388e-01,-0.003626881,0.97505554


Checking DMR

In [6]:
library(bumphunter)
ls()

[1] "allergy_status"      "ann"                 "batch"              
 [4] "data_dir"            "design_189148"       "fit_114134"         
 [7] "fit_189148"          "isg15"               "mVals_114134_shared"
[10] "mVals_189148_shared" "scan_dates"          "status_114134"      
[13] "status_189148"       "tt_114134"           "tt_189148"

In [7]:
chr <- ann[rownames(mVals_114134_shared), 'chr']
pos <- ann[rownames(mVals_114134_shared), 'pos']

sum(is.na(chr))   #should be 0
ord <- order(chr, pos)
cluster <- clusterMaker(chr[ord], pos[ord], maxGap = 500)
length(unique(cluster))

[1] 0

[1] 381976

In [8]:
rm(mVals_189148_shared, fit_114134, fit_189148, tt_189148)
gc()
chr <- ann[rownames(mVals_114134_shared), 'chr']
pos <- ann[rownames(mVals_114134_shared), 'pos']
rm(ann); gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,16938975,904.7,27771187,1483.2,17786617,950.0
Vcells,165876058,1265.6,474691389,3621.7,474664072,3621.4


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,16939615,904.7,27771187,1483.2,17786617,950.0
Vcells,165877230,1265.6,474691389,3621.7,474664072,3621.4


In [9]:
m_ord <- mVals_114134_shared[ord, ]
rm(mVals_114134_shared); gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,16939575,904.7,27771187,1483.2,17786617,950.0
Vcells,165877150,1265.6,474691389,3621.7,474664072,3621.4


In [ ]:
design_114134 <- model.matrix(~ status_114134)

chr_ord <- chr[ord]
pos_ord <- pos[ord]

set.seed(42)
bumps <- bumphunter(m_ord, design_114134, chr = chr_ord, pos = pos_ord,
                    cluster = cluster, coef = 2,
                    cutoff = 0.15, B = 250, type = "M")

saveRDS(bumps, file.path(data_dir, 'bumphunter_114134.rds'))
dim(bumps$table)

[bumphunterEngine] Using a single core (backend: doSEQ, version: 1.5.2).

[bumphunterEngine] Computing coefficients.

[bumphunterEngine] Performing 250 permutations.

[bumphunterEngine] Computing marginal permutation p-values.

[bumphunterEngine] cutoff: 0.15

[bumphunterEngine] Finding regions.

[bumphunterEngine] Found 38638 bumps.

[bumphunterEngine] Computing regions for each permutation.

Loading required package: rngtools

[bumphunterEngine] Estimating p-values and FWER.



In [ ]:
#Checking significant (how many regions make it past FWER correction)
sum(bumps$table$fwer < 0.05, na.rm = TRUE)
sum(bumps$table$p.valueArea < 0.05, na.rm = TRUE)

In [ ]:
subset(bumps$table, chr == 'chr1' & start < 950000 & end > 948000) #Searching manually for ISG15 bc its fwer is too big

In [ ]:
bumps$table[1:10, c('chr','start','end','value','area','L','p.value','fwer')]

Check Relevant Probes

In [ ]:
#reload annotation bc it was deliberately removed earlier to free memory
library(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)
ann <- getAnnotation(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)

In [ ]:
top_probes <- rownames(ann)[ann$chr == 'chr5' &
                            ann$pos >= 135414858 &
                            ann$pos <= 135416613]
length(top_probes)
unique(ann[top_probes, 'UCSC_RefGene_Name'])

ann_top <- ann[chr == 'chr5' & pos >= 135414858 & pos <= 135416613, ]
unique(ann_top$UCSC_RefGene_Name)

In [ ]:
probes5 <- rownames(ann)[ann$chr=='chr5' & ann$pos>=135414858 & ann$pos<=135416613] #Checking regional distribution of differences 
probes5 <- intersect(probes5, rownames(m_ord))
b <- 2^m_ord[probes5,]/(1+2^m_ord[probes5,])
hist(colMeans(b), breaks=20, main='chr5 region, mean beta per sample')
table(colMeans(b) > 0.5, status_114134) #Checking to see if it correlates with allergy status

In [ ]:
fisher.test(table(colMeans(b) > 0.5, status_114134)) #Chyecking if it correlates with allergy status

In [ ]:
data.frame(probe = probes5,
           lfc = tt_114134[probes5,'logFC'],
           p = tt_114134[probes5,'P.Value'])[order(tt_114134[probes5,'P.Value']),][1:5,]

In [ ]:
saveRDS(list(bumps_B50 = bumps$table, #Saving results
             nc886_probes = probes5,
             nc886_fisher = fisher.test(table(colMeans(b) > 0.5, status_114134)),
             nc886_modes = "bimodal, ~0.2 and ~0.55"),
        file.path(data_dir, 'dmr_results.rds'))

Reverse and Mirror Direction Probe Check (GSE118148-->GSE1124134)

In [ ]:
top_b <- rownames(tt_189148)[order(tt_189148$P.Value)][1:100] #Taking first 100 most significant

comp <- data.frame(
  probe = top_b,
  lfc_adol   = tt_189148[top_b, 'logFC'],
  p_adol     = tt_189148[top_b, 'P.Value'],
  lfc_infant = tt_114134[top_b, 'logFC'],
  p_infant   = tt_114134[top_b, 'P.Value']
)

comp$concordant <- sign(comp$lfc_adol) == sign(comp$lfc_infant) #Checking to see if direction is shared

table(comp$concordant)
binom.test(sum(comp$concordant), 100, p = 0.5)

In [ ]:
top_a <- rownames(tt_114134)[order(tt_114134$P.Value)][1:100] #Other way (GSE114134-->GSE189148)

comp_a <- data.frame(
  probe = top_a,
  lfc_infant = tt_114134[top_a, 'logFC'],
  lfc_adol   = tt_189148[top_a, 'logFC'],
  p_adol     = tt_189148[top_a, 'P.Value']
)
comp_a$concordant <- sign(comp_a$lfc_infant) == sign(comp_a$lfc_adol)

table(comp_a$concordant)
binom.test(sum(comp_a$concordant), 100, p = 0.5)

In [ ]:
#Permuation check on infant-->adolescent
set.seed(42)
perm_conc <- replicate(200, {
  y_shuf <- sample(status_114134)
  f <- eBayes(lmFit(m_ord, model.matrix(~ y_shuf)))
  t <- topTable(f, coef = 2, number = Inf, sort.by = 'none')
  tp <- rownames(t)[order(t$P.Value)][1:100]
  mean(sign(t[tp, 'logFC']) == sign(tt_189148[tp, 'logFC']))
})
mean(perm_conc); quantile(perm_conc, c(0.025, 0.975))

In [ ]:
saveRDS(list(obs_infant_to_adol = mean(comp_a$concordant),
             obs_adol_to_infant = mean(comp$concordant),
             binom_p = binom.test(sum(comp_a$concordant), 100, p = 0.5)$p.value,
             perm_null = perm_conc,
             perm_mean = mean(perm_conc),
             perm_ci = quantile(perm_conc, c(0.025, 0.975))),
        file.path(data_dir, 'directional_concordance.rds'))

Bumphunter on GSE118148

In [ ]:
library(bumphunter)
library(doParallel)
registerDoParallel(cores = 4)

mVals_189148_shared <- readRDS(file.path(data_dir, 'mVals_189148_shared.rds'))

chr_b <- ann[rownames(mVals_189148_shared), 'chr']
pos_b <- ann[rownames(mVals_189148_shared), 'pos']
stopifnot(sum(is.na(chr_b)) == 0)

ord_b   <- order(chr_b, pos_b)
m_ord_b <- mVals_189148_shared[ord_b, ]
chr_ord_b <- chr_b[ord_b]
pos_ord_b <- pos_b[ord_b]

cluster_b <- clusterMaker(chr_ord_b, pos_ord_b, maxGap = 500)
length(unique(cluster_b))

This cell shows the exact bumphunter() call used for the adolescent cohort's original run, included here for documentation. The boostrap run was executed as a standalone script (rerun_bootstrap.R) via Rscript from the command line. The corresponding permutation-based result (bumphunter_189148_B250.rds, loaded below) was generated the same way in an earlier session, using this identical call with the nullMethod argument removed (reverting to bumphunter's default). Results from both are compared in the cells that follow.

In [ ]:
design_189148_bh <- model.matrix(~ status_189148 + factor(scan_dates))

set.seed(42)
t0 <- Sys.time()
bumps_b <- bumphunter(m_ord_b, design_189148_bh,
                      chr = chr_ord_b, pos = pos_ord_b,
                      cluster = cluster_b, coef = 2,
                      cutoff = 0.15, B = 250, type = "M")
Sys.time() - t0

saveRDS(bumps_b, file.path(data_dir, 'bumphunter_189148_B250_bootstrap.rds'))
dim(bumps_b$table)

In [ ]:
nrow(bumps_b$table)
sum(bumps_b$table$fwer < 0.05, na.rm = TRUE)

In [ ]:
saveRDS(bumps_b, file.path(data_dir, 'bumphunter_189148_B250.rds')) #Saving in original name (this was the non-bootstrap version)

In [ ]:
bumps_b_bootstrap <- readRDS(file.path(data_dir, 'bumphunter_189148_B250_bootstrap.rds'))

isg15_bootstrap <- subset(bumps_b_bootstrap$table, chr == 'chr1' & start < 950000 & end > 948000)
rgs14_bootstrap <- subset(bumps_b_bootstrap$table, chr == 'chr5' & start < 176799000 & end > 176797000)

cat("ISG15 — bootstrap:\n")
print(isg15_bootstrap[, c('chr','start','end','value','p.value','fwer')])

cat("\nISG15 — original permutation, for comparison:\n")
print(subset(bumps_b$table, chr == 'chr1' & start < 950000 & end > 948000)[, c('chr','start','end','value','p.value','fwer')])

cat("\nRGS14 — bootstrap:\n")
print(rgs14_bootstrap[, c('chr','start','end','value','p.value','fwer')])

cat("\nRGS14 — original permutation, for comparison:\n")
print(subset(bumps_b$table, chr == 'chr5' & start < 176799000 & end > 176797000)[, c('chr','start','end','value','p.value','fwer')])

- Two adjacent CpG probes near ISG15 (cg08469540, cg25610492) show direction-matched hypomethylation in food-allergic individuals in both the infant (GSE114134) and adolescent (GSE189148) cohorts.
- The infant cohort's strongest genome-wide hit by raw p-value (chr5, nc886/VTRNA2-1) was checked and excluded as a known imprinted, bimodally-methylated locus unrelated to allergy status (Fisher's exact test, p=0.585)
- An unbiased top-100-probe concordance check found the infant cohort's top differentially methylated probes replicate direction in the adolescent cohort at a rate (67%) significantly above a permutation-based null (~51%, p=0.00087), which is evidence of a broader, diffuse cross-cohort signal beyond any single candidate gene. The reverse direction (adolescent top-100 checked in infant) was not significant, though (51%),  likely because the infant cohort's larger sample size gave a more stable top-100 ranking.
- The adolescent cohort alone has zero bumphunter regions clearing genome-wide FWER < 0.05.
- The adolescent-cohort bumphunter run was re-run with nullMethod="bootstrap" (run externally via rerun_bootstrap.R) to address a warning about permutation nulls being unreliable with multiple design-matrix covariates; results were essentially unchanged from the original permutation-based run (ISG15: p=0.00589 → 0.00617; RGS14: p=0.00110 → 0.00118)
- The infant cohort's original B=50 permutation count was updated to B=250 to match the adolescent cohort.

Continued in notebook 9 (genome-wide concordance scan) and notebook 10 (RNA-seq cross-validation).